# Two-Arm Bandit: Latent Space Visualisation

Loads a pre-trained BundleNet run (from a grid search) and visualises the 3-D latent
trajectories from six canonical viewpoints plus one isometric view.  
Also includes basic dataset sanity checks: sparsity, state transitions, and unique behavioural states.


In [1]:
import json
import os
from pathlib import Path

import numpy as np
from PIL import Image

from ncmcm.data_loaders.bandit_task import BanditTaskNeuroPixelsDataset
from ncmcm.visualisers.latent_space import LatentSpaceVisualiser


## Configuration

Set the session and grid-search run paths here before running the notebook.


In [2]:
# --- paths ---
data_path = Path("./JPAS_0023_20230922")
run_dir = Path(
    "./results/grid_search_20260108_205529"
    "/run_000_data_path=JPAS_0023_20230922_downsample_fs=30_downsample_method=gaussian"
    "_good_neurons_only=False_apply_hold_transitions=False_normalize_method=minmax_global"
    "_window=90_latent_dim=3_batch_size=50_learning_rate=5e-05_gamma=0.75"
)
figures_dir = Path("./figures") / run_dir.name
figures_dir.mkdir(parents=True, exist_ok=True)

# --- dataset parameters (should match the run) ---
downsample_fs = 15
downsample_method = "count"
good_neurons_only = True

# --- figure crop margin (cm on each side) ---
crop_cm = 4


## Dataset Sanity Checks


In [3]:
dataset = BanditTaskNeuroPixelsDataset(
    data_path=data_path,
    downsample_fs=downsample_fs,
    downsample_method="gaussian",
    good_neurons_only=good_neurons_only,
)

n_neurons, n_timepoints = dataset.x.shape
recording_minutes = n_timepoints / dataset.fs / 60
print(f"Neurons: {n_neurons},  Timepoints: {n_timepoints},  Duration: {recording_minutes:.1f} min")


Dataset cached to: JPAS_0023_20230922/BanditTaskNeuroPixelsDataset/fs15_gaussian_good_normNone_c5e7cd4a.pkl
Loaded BanditTaskNeuroPixelsDataset from JPAS_0023_20230922
Neuronal data shape: (367, 19397), Behavioral data shape: (1, 19397), Sampling frequency: 14.999287789475963 Hz
Behavioral labels: {0: 'intertrial', 1: 'hold', 2: 'reward', 3: 'no reward', 4: 'choosing left', 5: 'choosing right'}
Neurons: 367,  Timepoints: 19397,  Duration: 21.6 min


In [4]:
dataset.check_state_transitions()


{'valid': False,
 'observed_transitions': {('intertrial', 'hold'): 237,
  ('hold', 'choosing right'): 136,
  ('choosing right', 'reward'): 86,
  ('reward', 'intertrial'): 157,
  ('choosing right', 'no reward'): 50,
  ('no reward', 'intertrial'): 79,
  ('hold', 'choosing left'): 100,
  ('choosing left', 'no reward'): 29,
  ('choosing left', 'reward'): 71,
  ('hold', 'no reward'): 1},
 'invalid_transitions': [{'from': 'hold',
   'to': 'no reward',
   'count': 1,
   'indices': [np.int64(1852)]}]}

In [5]:
# Spike-matrix sparsity
total = n_neurons * n_timepoints
nonzeros = dataset.x.count_nonzero()
pct_nonzero = nonzeros / total * 100
print(f"Total entries : {total:,}")
print(f"Non-zero      : {nonzeros:,}  ({pct_nonzero:.4f} %)")
print(f"Zero (silent) : {total - nonzeros:,}  ({100 - pct_nonzero:.4f} %)")


Total entries : 7,118,699
Non-zero      : 2,870,652  (40.3255 %)
Zero (silent) : 4,248,047  (59.6745 %)


In [6]:
# Unique behavioural states from metrics.json
with open(data_path / "metrics.json") as f:
    metrics = json.load(f)

unique_states = list(dict.fromkeys(name for _, name in metrics["metrics"]["states"]))
print(f"Unique states ({len(unique_states)}): {unique_states}")


Unique states (7): ['delay', 'waiting', 'intertrial', 'hold', 'choosing', 'reward', 'no reward']


## Latent Space Visualisation

Load the latent trajectories saved by the grid-search run and render the 3-D phase-space
from six canonical viewpoints (front/back/top/bottom/left/right) plus one isometric view.
Each figure is saved as a PNG and cropped by `crop_cm` centimetres on every side to remove whitespace.


In [ ]:
# Load latent trajectories and matching behaviour labels
y = np.load(run_dir / "/home/kerim/Projects/Neural Algorithms/NC-MCM/datasets/raw/twoArmBandit/results/archive_20260212/?_grid_search_20260107_005655/?_run_000_data_path=JPAS_0023_20230922_downsample_fs=25_downsample_method=gaussian_good_neurons_only=False_apply_hold_transitions=False_normalize_method=minmax_window=50_latent_dim=3_batch_size=50_learning_rate=0.0001_gamma=0.75/data/latent_trajectories_train.npy")  # (T, latent_dim)
b = np.load(run_dir / "/home/kerim/Projects/Neural Algorithms/NC-MCM/datasets/raw/twoArmBandit/results/archive_20260212/?_grid_search_20260107_005655/?_run_000_data_path=JPAS_0023_20230922_downsample_fs=25_downsample_method=gaussian_good_neurons_only=False_apply_hold_transitions=False_normalize_method=minmax_window=50_latent_dim=3_batch_size=50_learning_rate=0.0001_gamma=0.75/data/behavior_labels_train.npy")      # (T,)

# Load colour map from a fresh (minimal) dataset instance
ds_colors = BanditTaskNeuroPixelsDataset(
    data_path=data_path,
    downsample_fs=downsample_fs,
    downsample_method=downsample_method,
    good_neurons_only=good_neurons_only,
    normalize_method=None,
)
b_labels = ds_colors.b_labels
b_colors_rgb = ds_colors.get_rgb_colors_for_visualizer()
del ds_colors

vis = LatentSpaceVisualiser(y, b, b_labels, show_points=True, colors=b_colors_rgb, legend=False)

views = [
    ((0, 180), "back"),
    ((-90, 0), "bottom"),
    ((0, 0),   "front"),
    ((90, 0),  "top"),
    ((0, 90),  "right"),
    ((0, -90), "left"),
    ((30, 45), "iso"),
]

for (elev, azim), name in views:
    fname = figures_dir / f"latent_space_{name}.png"
    fig, ax = vis.plot_phase_space(show_fig=False, axis_view=(elev, azim), filename=str(fname))

    with Image.open(fname) as im:
        dpi = im.info.get("dpi", (100, 100))[0]
        pad_px = int(round((crop_cm / 2.54) * dpi))
        w, h = im.size
        box = (pad_px, pad_px, w - pad_px, h - pad_px)
        if box[2] > box[0] and box[3] > box[1]:
            im.crop(box).save(fname)
            print(f"Saved {fname.name}  ({box[2]-box[0]}×{box[3]-box[1]} px)")
        else:
            print(f"Skipping {fname.name}: crop margin too large")


Dataset cached to: JPAS_0023_20230922/BanditTaskNeuroPixelsDataset/fs15_count_good_normNone_8507c6b4.pkl
Loaded BanditTaskNeuroPixelsDataset from JPAS_0023_20230922
Neuronal data shape: (367, 19397), Behavioral data shape: (1, 19397), Sampling frequency: 14.999287789475963 Hz
Behavioral labels: {0: 'intertrial', 1: 'hold', 2: 'reward', 3: 'no reward', 4: 'choosing left', 5: 'choosing right'}
Saved latent_space_back.png  (1456×1456 px)
Saved latent_space_bottom.png  (1456×1456 px)
Saved latent_space_front.png  (1456×1456 px)
Saved latent_space_top.png  (1456×1456 px)
